In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np

from src.utils import (
    get_args,
    set_seed,
    get_datesets_and_loaders,
    get_trained_VAE,
    get_trained_VAE_with_domain_classifier,
    get_trained_classifier,
    get_trained_classifier_Base,
    test_model,
    prepare_report,
    run_all_senario
)
from src.tupl import run_tupl
from src.our_tupl import GENERATION_POLICIES, run_m1_tupl, run_all_senario_m1_tupl
from src.vista_gzsda import run_vista

/home/asad/workspace/DomainProject/changeDomain/notebooks/effective-gzsda/gzsda/src/utils.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy
/home/asad/workspace/anaconda3/envs/asad/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

DOMAIN_SET = ['angry', 'childlike', 'depressed', 'neutral', 'old', 'proud', 'strutting']
DATA_DIR = './data/ActionStyleDataset/'
DATASET_DETAILS = {
    'prefix': 'ActionStyle-',
    'suffix': '-clip.mat',
    'resnet_feature': 'clip_features',
    'split_file_name': 'instanceSplit_actionStyle_unseen2.mat',
}
NUM_LABELS = 5

In [4]:
import sys

sys.argv.extend([
    "--encoder_layer_sizes", "512", "512",
    "--decoder_layer_sizes", "512", "512",
])

In [5]:
import json
from pathlib import Path

RESULT_OBJ_PATH = "./result/json/actionStyle.json"
RESULT_CSV_PATH = "./result/csv/actionStyle.csv"
path = Path(RESULT_OBJ_PATH)

if path.exists():
    with path.open("r", encoding="utf-8") as f:
        result = json.load(f)
else:
    result = {}

result.keys()

dict_keys(['base', 'CCVAE', 'our0', 'our_GRE', 'TUPL', 'our_TUPL_real_plus_src2tgt', 'our_TUPL_real_plus_src2tgt_unseen', 'our_TUPL_interp_src2tgt', 'VisTA'])

In [6]:
base = "base"
CCVAE = "CCVAE"
our0 = "our0"
our_GRE = "our_GRE"
tupl = "TUPL"
our_tupl = "our_TUPL"
vista = "VisTA"

# clear last result
# result.pop(base, None)
# result.pop(CCVAE, None)
# result.pop(our0, None)
# result.pop(our_GRE, None)
# result.pop(tupl, None)
# result.pop(vista, None)
# for k in [f"{our_tupl}_{p}" for p in GENERATION_POLICIES]:
#     result.pop(k, None)

## Base

In [7]:
def main_base(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    classifier = get_trained_classifier_Base(
        data_loaders=data_loaders,
        NUM_LABELS=NUM_LABELS,
        device=device,
        input_dim=512)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [8]:
if base not in result:
    result[base] = run_all_senario(main_base, DOMAIN_SET, input_dim=512, num_trial=6)

# GZSDA

In [9]:
def main_gzsda(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        input_dim=512)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [10]:
if CCVAE not in result:
    result[CCVAE] = run_all_senario(main_gzsda, DOMAIN_SET, input_dim=512, num_trial=6)

## m0

In [11]:
def main_m0(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30,
        input_dim=512)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [12]:
if our0 not in result:
    result[our0] = run_all_senario(main_m0, DOMAIN_SET, input_dim=512, num_trial=6)

## m1: seperate after encoder

In [13]:
def main_m1(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE_with_domain_classifier(
        data_loaders=data_loaders,
        args=args,
        device=device)
        
    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30,
        input_dim=512)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [14]:
if our_GRE not in result:
    result[our_GRE] = run_all_senario(main_m1, DOMAIN_SET, input_dim=512, num_trial=6)

## TUPL

In [15]:
def main_tupl(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    acc_s, acc_u, h = run_tupl(
        data_root="./data/",
        dataset="actionstyle",
        source=args.sourceDomainIndex,
        target=args.targetDomainIndex,
        trial=args.trialIndex,
        seed=args.seed,
        device=device,
        quiet=True,
        return_model=False,
    )
    print('seen acc:{:2.4f}, unseen acc:{:2.4f}, H:{:2.4f}'.format(acc_s / 100, acc_u / 100, h / 100))
    return None, None, acc_s / 100.0, acc_u / 100.0

In [16]:
if tupl not in result:
    result[tupl] = run_all_senario(main_tupl, DOMAIN_SET, input_dim=512, num_trial=6)

## our_TUPL: m1 VAE + TUPL

In [17]:
def main_m1_tupl(args, policy="real_plus_src2tgt"):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    acc_s, acc_u, h = run_m1_tupl(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS,
        policy=policy,
        device=device,
        quiet=True,
    )
    print('seen acc:{:2.4f}, unseen acc:{:2.4f}, H:{:2.4f}'.format(acc_s / 100, acc_u / 100, h / 100))
    return None, None, acc_s / 100.0, acc_u / 100.0

In [18]:
missing = [p for p in GENERATION_POLICIES if f"{our_tupl}_{p}" not in result]
if missing:
    result.update(run_all_senario_m1_tupl(
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS,
        policies=missing,
        input_dim=512,
        num_trial=6,
    ))

## VisTA

Feature-space VisTA on MotionCLIP embeddings (no images, no Grad-CAM VAC).
Train: labeled source (all classes) + unlabeled target seen-train.
Test: Acc_s / Acc_u / H-mean on the GZSDA target test split.

In [19]:
def main_vista(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    return run_vista(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS,
        device=device,
        quiet=True,
    )

In [20]:
if vista not in result:
    result[vista] = run_all_senario(main_vista, DOMAIN_SET, input_dim=512, num_trial=6)


## Merge results

In [21]:
with open(RESULT_OBJ_PATH, "w") as f:
    json.dump(result, f, indent=2)

In [22]:
# ignore our0
_ = result.pop(our0, None)
_ = result.pop("our_TUPL_real_plus_src2tgt", None)
_ = result.pop("our_TUPL_real_plus_src2tgt_unseen", None)
# _ = result.pop("our_TUPL_interp_src2tgt", None)

In [23]:
import pandas as pd
import re

# Display all rows in the dataframe by setting display.max_rows to None
# pd.set_option('display.max_rows', None)

rows = [(k, m, result[m][k]) for m in result for k in result[m]]
df = pd.DataFrame(rows, columns=['domain', 'method', 'values'])

def extract_metrics(text):
    matches = dict(re.findall(r'(\w+):\s+([\d.]+\s*±\s*[\d.]+)', text))
    return pd.Series(matches)

df[['seen', 'unseen', 'H-mean']] = df['values'].apply(extract_metrics)
df = df[['domain', 'method', 'seen', 'unseen', 'H-mean']]

df['method'] = pd.Categorical(
    df['method'],
    categories=[base, vista, CCVAE, tupl, our0, our_GRE] + [f"{our_tupl}_{p}" for p in GENERATION_POLICIES],
    ordered=True,
)
df = df.sort_values(['domain', 'method']).reset_index(drop=True)

df

,domain,method,seen,unseen,H-mean
0,angry -> childlike,base,93.40 ± 2.82,30.35 ± 11.40,38.87 ± 12.93
1,angry -> childlike,VisTA,75.00 ± 9.68,53.22 ± 10.54,54.64 ± 6.44
2,angry -> childlike,CCVAE,90.62 ± 4.19,26.72 ± 11.77,32.82 ± 14.12
3,angry -> childlike,TUPL,91.29 ± 2.74,59.18 ± 4.78,71.17 ± 3.43
4,angry -> childlike,our_GRE,85.76 ± 6.49,48.19 ± 12.44,54.82 ± 11.97
...,...,...,...,...,...
247,strutting -> proud,VisTA,61.11 ± 14.34,35.98 ± 9.72,31.46 ± 7.52
248,strutting -> proud,CCVAE,66.67 ± 10.54,28.95 ± 15.51,27.52 ± 12.62
249,strutting -> proud,TUPL,98.96 ± 1.04,44.60 ± 13.04,56.24 ± 12.01
250,strutting -> proud,our_GRE,64.58 ± 9.36,49.12 ± 12.92,46.13 ± 9.70


In [24]:
df.to_csv(RESULT_CSV_PATH, index=False)

In [25]:
# Average over target domain and keep separate methods
df_copy = df.copy()
df_copy['target'] = df_copy['domain'].str.split('->').str[1].str.strip()
target_method_avg = (
    df_copy.groupby(['target', 'method'], observed=False)[['seen', 'unseen', 'H-mean']]
    .apply(lambda g: g.replace(r'±.*', '', regex=True).astype(float).mean().round(2))
)

target_method_avg = target_method_avg.dropna()
target_method_avg.to_csv("result/csv/actionStyle_target_method_avg.csv")
target_method_avg

seen  unseen  H-mean
target    method                                        
angry     base                     85.88   31.90   37.60
          VisTA                    59.54   45.02   38.32
          CCVAE                    81.69   27.11   31.47
          TUPL                     98.71   49.81   61.74
          our_GRE                  77.88   38.47   41.34
          our_TUPL_interp_src2tgt  97.45   52.35   63.38
childlike base                     80.61   27.88   32.44
          VisTA                    58.40   37.81   35.96
          CCVAE                    76.03   23.73   26.60
          TUPL                     93.36   51.46   62.92
          our_GRE                  71.11   37.31   37.86
          our_TUPL_interp_src2tgt  88.64   48.53   59.04
depressed base                     84.90   16.62   21.66
          VisTA                    49.67   39.49   29.41
          CCVAE                    79.83   14.21   18.05
          TUPL                     93.21   37.85   50.14
          our_GRE                  76.60   31.82   35.25
          our_TUPL_interp_src2tgt  90.70   38.42   50.20
neutral   base                     86.38   34.05   39.65
          VisTA                    59.84   42.66   38.31
          CCVAE                    83.30   30.27   35.11
          TUPL                     97.00   51.43   63.30
          our_GRE                  79.30   40.17   42.76
          our_TUPL_interp_src2tgt  91.62   50.22   60.46
old       base                     85.98   16.92   22.19
          VisTA                    51.28   33.85   25.38
          CCVAE                    83.71   12.81   17.46
          TUPL                     91.20   42.29   54.14
          our_GRE                  79.92   33.02   36.78
          our_TUPL_interp_src2tgt  89.08   36.28   48.29
proud     base                     89.14   32.63   40.30
          VisTA                    58.15   44.06   37.16
          CCVAE                    86.18   26.34   32.58
          TUPL                     97.23   58.10   69.67
          our_GRE                  84.27   42.28   47.18
          our_TUPL_interp_src2tgt  95.89   52.65   64.66
strutting base                     83.52   26.21   32.22
          VisTA                    54.76   40.75   33.87
          CCVAE                    80.19   22.80   28.62
          TUPL                     95.40   43.81   55.14
          our_GRE                  77.71   34.49   38.25
          our_TUPL_interp_src2tgt  94.05   49.33   60.27

In [26]:
df["H-mean_value"] = (
    df["unseen"]
    # df["H-mean"]
    .str.split("±")
    .str[0]
    .astype(float)
)
df

,domain,method,seen,unseen,H-mean,H-mean_value
0,angry -> childlike,base,93.40 ± 2.82,30.35 ± 11.40,38.87 ± 12.93,30.35
1,angry -> childlike,VisTA,75.00 ± 9.68,53.22 ± 10.54,54.64 ± 6.44,53.22
2,angry -> childlike,CCVAE,90.62 ± 4.19,26.72 ± 11.77,32.82 ± 14.12,26.72
3,angry -> childlike,TUPL,91.29 ± 2.74,59.18 ± 4.78,71.17 ± 3.43,59.18
4,angry -> childlike,our_GRE,85.76 ± 6.49,48.19 ± 12.44,54.82 ± 11.97,48.19
...,...,...,...,...,...,...
247,strutting -> proud,VisTA,61.11 ± 14.34,35.98 ± 9.72,31.46 ± 7.52,35.98
248,strutting -> proud,CCVAE,66.67 ± 10.54,28.95 ± 15.51,27.52 ± 12.62,28.95
249,strutting -> proud,TUPL,98.96 ± 1.04,44.60 ± 13.04,56.24 ± 12.01,44.60
250,strutting -> proud,our_GRE,64.58 ± 9.36,49.12 ± 12.92,46.13 ± 9.70,49.12


In [27]:
best = df.loc[df.groupby("domain")["H-mean_value"].idxmax()]
best = best[["domain", "method", "H-mean"]].reset_index(drop=True)

best

,domain,method,H-mean
0,angry -> childlike,TUPL,71.17 ± 3.43
1,angry -> depressed,TUPL,65.43 ± 5.17
2,angry -> neutral,our_TUPL_interp_src2tgt,74.01 ± 4.51
3,angry -> old,our_GRE,53.40 ± 12.43
4,angry -> proud,TUPL,74.74 ± 10.18
5,angry -> strutting,TUPL,69.71 ± 9.27
6,childlike -> angry,our_TUPL_interp_src2tgt,67.82 ± 14.83
7,childlike -> depressed,our_TUPL_interp_src2tgt,60.15 ± 10.72
8,childlike -> neutral,TUPL,75.81 ± 6.15
9,childlike -> old,TUPL,72.11 ± 3.75


In [28]:
from collections import Counter
Counter(best.method)

Counter({'TUPL': 18, 'our_TUPL_interp_src2tgt': 16, 'our_GRE': 6, 'VisTA': 2})